# 🚴 M2_01 Solutions: Amsterdam Bike Data Acquisition

**Complete solutions with explanations**

---

## 📋 How to Use This Notebook

1. **Try first!** Attempt each task in the main notebook before checking solutions
2. **Learn from differences**: Compare your approach with the solution
3. **Understand, don't copy**: Read the explanations and comments

---

## Setup (Same as Main Notebook)

In [ ]:
import os
import json
from datetime import datetime
from pathlib import Path
import pandas as pd
import requests
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
%matplotlib inline

In [ ]:
# Configuration
BASE_URL = "http://api.citybik.es/v2"
NETWORK_ID = "ov-fiets"
NETWORK_URL = f"{BASE_URL}/networks/{NETWORK_ID}"

DATA_DIR = Path("../../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_FILE = DATA_DIR / f"amsterdam_bike_{timestamp}.json"

---

## ✅ Task 1 Solution: Basic API Request

In [ ]:
# SOLUTION: Task 1 - Fetch bike data

print("📡 Fetching bike data from CityBikes API...")
print(f"🔗 URL: {NETWORK_URL}\n")

# Make GET request
response = requests.get(NETWORK_URL)

# Check if successful
if response.status_code == 200:
    # Parse JSON
    data = response.json()
    
    # Extract information
    network_name = data['network']['name']
    city = data['network']['location']['city']
    country = data['network']['location']['country']
    num_stations = len(data['network']['stations'])
    
    # Print results
    print(f"✅ Success! Fetched data for:")
    print(f"   Network: {network_name}")
    print(f"   Location: {city}, {country}")
    print(f"   Stations: {num_stations}")
else:
    print(f"❌ Failed: HTTP {response.status_code}")

# Key Learning Points:
# - requests.get() returns a Response object
# - status_code 200 means success
# - .json() parses response into Python dict
# - Navigate nested JSON with dictionary keys

### Alternative Approach: Using try/except

In [ ]:
# Alternative: More defensive coding
try:
    response = requests.get(NETWORK_URL, timeout=10)
    response.raise_for_status()  # Raises exception for 4xx/5xx
    data = response.json()
    print(f"✅ Fetched {len(data['network']['stations'])} stations")
except requests.exceptions.RequestException as e:
    print(f"❌ Error: {e}")

---

## ✅ Task 2 Solution: JSON Exploration

In [ ]:
# SOLUTION: Task 2 - Explore JSON structure

# 1. Top-level keys
print("📂 Top-level keys:")
print(f"   {list(data.keys())}\n")

# 2. Network metadata keys
print("🔑 Network metadata keys:")
print(f"   {list(data['network'].keys())}\n")

# 3. First station structure
first_station = data['network']['stations'][0]
print("🚴 First station keys:")
print(f"   {list(first_station.keys())}\n")

# 4. First station full details
print("📋 First station details:")
print(json.dumps(first_station, indent=2))

# Learning: Understanding JSON structure is crucial
# before converting to DataFrame!

---

## ✅ Task 3 Solution: Error Handling

In [ ]:
# SOLUTION: Task 3 - Comprehensive error handling

def fetch_bike_data_safe(network_id, timeout=10):
    """
    Fetch bike data with comprehensive error handling.
    
    Parameters:
    -----------
    network_id : str
        The network ID (e.g., 'ns-bikes')
    timeout : int
        Request timeout in seconds
    
    Returns:
    --------
    dict or None
        JSON data if successful, None otherwise
    """
    url = f"{BASE_URL}/networks/{network_id}"
    
    try:
        # Make request with timeout
        response = requests.get(url, timeout=timeout)
        
        # Raise exception for bad HTTP status
        response.raise_for_status()
        
        # Parse and return JSON
        return response.json()
        
    except requests.exceptions.Timeout:
        print(f"⏱️ Timeout: Request took longer than {timeout} seconds")
        return None
        
    except requests.exceptions.ConnectionError:
        print("🔌 Connection Error: Could not connect to API")
        return None
        
    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP Error {e.response.status_code}: {e}")
        return None
        
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Request Error: {e}")
        return None
        
    except json.JSONDecodeError:
        print("📝 JSON Error: Could not parse response")
        return None


# Test with valid network
print("Test 1: Valid network")
result = fetch_bike_data_safe(NETWORK_ID)
if result:
    print(f"✅ Success! Fetched {len(result['network']['stations'])} stations\n")

# Test with invalid network
print("Test 2: Invalid network")
result = fetch_bike_data_safe("fake-network-12345")
if result is None:
    print("✅ Correctly handled invalid network\n")

# Test with deprecated/broken endpoint (ns-bikes)
print("Test 3: Deprecated endpoint (ns-bikes)")
print("⚠️ This endpoint no longer works - demonstrating real API failure:")
result = fetch_bike_data_safe("ns-bikes")
if result is None:
    print("✅ Correctly handled broken endpoint\n")
    print("💡 This is why we use 'ov-fiets' - it's a working endpoint!")

# Key Learning:
# - Always use specific exception types (most specific first)
# - Timeout is separate from ConnectionError
# - HTTPError catches 4xx and 5xx responses
# - RequestException is the catch-all
# - Return None on errors for easy checking

### Common Mistakes to Avoid

❌ **Mistake 1**: Using bare `except:` - too broad
```python
try:
    ...
except:  # BAD: catches everything including KeyboardInterrupt!
    pass
```

❌ **Mistake 2**: Wrong exception order
```python
except requests.exceptions.RequestException:  # BAD: catches all first
    ...
except requests.exceptions.Timeout:  # Never reached!
    ...
```

✅ **Correct**: Specific to general
```python
except requests.exceptions.Timeout:  # GOOD: specific first
    ...
except requests.exceptions.RequestException:  # Catch-all last
    ...
```

---

## ✅ Tasks 4.1-4.5 Solutions: DataFrame Conversion

In [ ]:
# Fetch data using safe function
data = fetch_bike_data_safe(NETWORK_ID)
if not data:
    raise ValueError("Failed to fetch data")

### Task 4.1: Extract and Convert

In [ ]:
# SOLUTION: Task 4.1 - Extract and convert to DataFrame

# Extract stations list
stations = data['network']['stations']

# Convert to DataFrame
df_bikes = pd.DataFrame(stations)

# Display info
print(f"✅ Created DataFrame: {df_bikes.shape[0]} rows × {df_bikes.shape[1]} columns\n")
print("First 3 rows:")
display(df_bikes.head(3))

print("\n" + "="*60)
print("DataFrame Info:")
print("="*60)
df_bikes.info()

### Task 4.2: Clean Columns and Parse Timestamps

In [ ]:
# SOLUTION: Task 4.2 - Clean column names and parse timestamps

# Parse timestamp to datetime
df_bikes['timestamp'] = pd.to_datetime(df_bikes['timestamp'])

# Rename columns for clarity
df_bikes = df_bikes.rename(columns={
    'free_bikes': 'bikes_available',
    'empty_slots': 'docks_available'
})

print("✅ Parsed timestamps and renamed columns")
print(f"\nTimestamp column type: {df_bikes['timestamp'].dtype}")
print(f"Columns: {list(df_bikes.columns)}")

### Task 4.3: Add Derived Columns

In [ ]:
# SOLUTION: Task 4.3 - Add derived columns

# Total capacity
df_bikes['total_capacity'] = df_bikes['bikes_available'] + df_bikes['docks_available']

# Utilization percentage
df_bikes['utilization_pct'] = (
    df_bikes['bikes_available'] / df_bikes['total_capacity'] * 100
).round(2)

# Empty flag
df_bikes['is_empty'] = df_bikes['bikes_available'] == 0

print("✅ Added derived columns")
print("\nSample values:")
display(df_bikes[['bikes_available', 'docks_available', 
                   'total_capacity', 'utilization_pct', 'is_empty']].head())

### Task 4.4: Add Network Metadata

In [ ]:
# SOLUTION: Task 4.4 - Add network metadata

df_bikes['network_id'] = data['network']['id']
df_bikes['network_name'] = data['network']['name']
df_bikes['city'] = data['network']['location']['city']
df_bikes['country'] = data['network']['location']['country']

print("✅ Added network metadata")
print(f"\nNetwork: {df_bikes['network_name'].iloc[0]}")
print(f"Location: {df_bikes['city'].iloc[0]}, {df_bikes['country'].iloc[0]}")

### Task 4.5: Reorder and Validate

In [ ]:
# SOLUTION: Task 4.5 - Reorder columns and validate

# Define desired column order
column_order = [
    'id', 'name', 'latitude', 'longitude',
    'bikes_available', 'docks_available', 'total_capacity', 'utilization_pct',
    'timestamp', 'network_id', 'network_name', 'city', 'country'
]

# Reorder columns
df_bikes = df_bikes[column_order]

print("✅ Reordered columns\n")
print("First 10 rows:")
display(df_bikes.head(10))

print("\n" + "="*60)
print("Summary Statistics:")
print("="*60)
display(df_bikes[['bikes_available', 'docks_available', 
                   'total_capacity', 'utilization_pct']].describe())

---

## ✅ Task 5.1 Solution: Summary Statistics

In [ ]:
# SOLUTION: Task 5.1 - Calculate summary statistics

print("📈 Summary Statistics:")
print("="*60)
display(df_bikes[['bikes_available', 'docks_available', 
                   'total_capacity', 'utilization_pct']].describe())

print("\n" + "="*60)
print("🚴 Bike Availability Summary:")
print("="*60)
print(f"Total Stations: {len(df_bikes)}")
print(f"Total Bikes Available: {df_bikes['bikes_available'].sum()}")
print(f"Total Docks Available: {df_bikes['docks_available'].sum()}")
print(f"Average Station Capacity: {df_bikes['total_capacity'].mean():.1f}")
print(f"Average Utilization: {df_bikes['utilization_pct'].mean():.1f}%")

print("\n" + "="*60)
print("⚠️ Problem Stations:")
print("="*60)
empty_stations = df_bikes[df_bikes['bikes_available'] == 0]
full_stations = df_bikes[df_bikes['docks_available'] == 0]
print(f"Empty stations (no bikes): {len(empty_stations)}")
print(f"Full stations (no docks): {len(full_stations)}")

# Key Insight: Understanding the data distribution helps
# identify operational issues (empty or full stations)

---

## ✅ Task 5.2 Solution: 4-Panel Dashboard

In [ ]:
# SOLUTION: Task 5.2 - Create 4-panel visualization

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('🚴 Amsterdam Bike Sharing - Real-time Overview', 
             fontsize=16, fontweight='bold')

# Panel 1: Histogram of bikes available
axes[0, 0].hist(df_bikes['bikes_available'], bins=20, 
                color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Bikes Available')
axes[0, 0].set_ylabel('Number of Stations')
axes[0, 0].set_title('Distribution of Bike Availability')
mean_bikes = df_bikes['bikes_available'].mean()
axes[0, 0].axvline(mean_bikes, color='red', linestyle='--', 
                   label=f"Mean: {mean_bikes:.1f}")
axes[0, 0].legend()

# Panel 2: Histogram of utilization
axes[0, 1].hist(df_bikes['utilization_pct'], bins=20, 
                color='darkorange', edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Utilization (%)')
axes[0, 1].set_ylabel('Number of Stations')
axes[0, 1].set_title('Station Utilization Distribution')
mean_util = df_bikes['utilization_pct'].mean()
axes[0, 1].axvline(mean_util, color='red', linestyle='--',
                   label=f"Mean: {mean_util:.1f}%")
axes[0, 1].legend()

# Panel 3: Top 10 stations
top10 = df_bikes.nlargest(10, 'bikes_available')[['name', 'bikes_available']]
axes[1, 0].barh(range(len(top10)), top10['bikes_available'], 
                color='green', alpha=0.7)
axes[1, 0].set_yticks(range(len(top10)))
axes[1, 0].set_yticklabels(top10['name'], fontsize=8)
axes[1, 0].set_xlabel('Bikes Available')
axes[1, 0].set_title('Top 10 Stations by Bike Availability')
axes[1, 0].invert_yaxis()

# Panel 4: Average capacity analysis
capacity_data = df_bikes[['bikes_available', 'docks_available']].mean()
axes[1, 1].bar(['Bikes\nAvailable', 'Docks\nAvailable'], capacity_data,
               color=['steelblue', 'coral'], alpha=0.7, edgecolor='black')
axes[1, 1].set_ylabel('Average Count')
axes[1, 1].set_title('Average Bikes vs Docks Available')
for i, v in enumerate(capacity_data):
    axes[1, 1].text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ 4-panel dashboard created")

---

## ✅ Task 5.3 Solution: Geographic Map

In [ ]:
# SOLUTION: Task 5.3 - Geographic station map

plt.figure(figsize=(12, 10))

# Create scatter plot
scatter = plt.scatter(
    df_bikes['longitude'], 
    df_bikes['latitude'],
    c=df_bikes['bikes_available'],
    s=df_bikes['total_capacity'] * 5,
    alpha=0.6,
    cmap='RdYlGn',  # Red=low, Green=high
    edgecolor='black',
    linewidth=0.5
)

# Add colorbar
plt.colorbar(scatter, label='Bikes Available')

# Labels and title
plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)
plt.title('🗺️ Amsterdam Bike Stations Map\n(Size = Capacity, Color = Availability)', 
          fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)

# BONUS: Annotate largest station
largest = df_bikes.nlargest(1, 'total_capacity').iloc[0]
plt.annotate(
    largest['name'],
    xy=(largest['longitude'], largest['latitude']),
    xytext=(10, 10), 
    textcoords='offset points',
    bbox=dict(boxstyle='round,pad=0.5', fc='yellow', alpha=0.7),
    arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0')
)

plt.tight_layout()
plt.show()

print("✅ Geographic map created")
print(f"\nLargest station: {largest['name']} ({largest['total_capacity']} capacity)")

---

## ✅ Task 5.4: Custom Analysis Examples

Here are three examples of custom analyses:

### Example 1: Utilization Categories

In [ ]:
# Custom Analysis 1: Station utilization categories

# Categorize stations
def categorize_utilization(pct):
    if pct < 20:
        return 'Very Low (<20%)'
    elif pct < 40:
        return 'Low (20-40%)'
    elif pct < 60:
        return 'Medium (40-60%)'
    elif pct < 80:
        return 'High (60-80%)'
    else:
        return 'Very High (>80%)'

df_bikes['util_category'] = df_bikes['utilization_pct'].apply(categorize_utilization)

# Plot
plt.figure(figsize=(10, 6))
category_counts = df_bikes['util_category'].value_counts()
colors = ['red', 'orange', 'yellow', 'lightgreen', 'green']
category_counts.plot(kind='bar', color=colors[:len(category_counts)], edgecolor='black')
plt.title('Station Utilization Categories', fontsize=14, fontweight='bold')
plt.xlabel('Utilization Category')
plt.ylabel('Number of Stations')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print(f"Most stations ({category_counts.max()}) fall in the '{category_counts.idxmax()}' category.")
print("This suggests balanced usage across the network.")

### Example 2: Capacity vs Location

In [ ]:
# Custom Analysis 2: Is station capacity related to location?

plt.figure(figsize=(10, 6))

# Scatter plot with size representing capacity
plt.scatter(df_bikes['longitude'], df_bikes['latitude'], 
           s=df_bikes['total_capacity']*10, 
           alpha=0.5, c='steelblue', edgecolor='black')

plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Station Capacity by Geographic Location\n(Larger circles = Higher capacity)', 
         fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate center point
center_lat = df_bikes['latitude'].mean()
center_lon = df_bikes['longitude'].mean()

# Distance from center
df_bikes['dist_from_center'] = (
    (df_bikes['latitude'] - center_lat)**2 + 
    (df_bikes['longitude'] - center_lon)**2
)**0.5

correlation = df_bikes['dist_from_center'].corr(df_bikes['total_capacity'])

print("\nInterpretation:")
print(f"Correlation between distance from center and capacity: {correlation:.3f}")
if abs(correlation) < 0.3:
    print("Weak correlation - capacity appears evenly distributed geographically.")
else:
    print("Notable correlation - central stations may have different capacity patterns.")

---

## 📚 Key Takeaways

### From Task 1-2 (API Basics)
- APIs return structured data (usually JSON)
- Always check status codes
- Explore JSON structure before DataFrame conversion

### From Task 3 (Error Handling)
- Use specific exception types
- Order exceptions from specific to general
- Return None for easy error checking
- Always include timeout parameter

### From Tasks 4.1-4.5 (DataFrame Operations)
- `pd.DataFrame()` directly converts list of dicts
- Parse timestamps with `pd.to_datetime()`
- Vectorized operations are fast and clean
- Derived columns add analytical value

### From Tasks 5.1-5.4 (Visualization)
- Start with summary statistics
- Subplots create comprehensive dashboards
- Geographic scatter plots reveal spatial patterns
- Custom analyses uncover business insights

---

**🎉 Well done! You now have production-ready API data acquisition skills!**